## Image Subtraction for Transient Detection

In [1]:
import os
import numpy as np
import os.path as pa
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from mpl_toolkits.axes_grid1 import ImageGrid
from photutils.background import Background2D, MedianBackground
from scipy.ndimage import gaussian_filter



from astropy.io import fits, ascii
from astropy.table import Table
from astropy.coordinates import SkyCoord
from astropy.visualization import astropy_mpl_style, ZScaleInterval
from astropy.nddata import Cutout2D
import astropy.units as u
from astropy.wcs import WCS
from regions import Regions


from sfft.EasySparsePacket import Easy_SparsePacket
from sfft.EasyCrowdedPacket import Easy_CrowdedPacket
from sfft.CustomizedPacket import Customized_Packet
from sfft.utils.pyAstroMatic.PYSWarp import PY_SWarp
import time
from pathlib import Path
import glob

import photometrus.photometry.photometry as photometry
import photometrus_utils as util
import sfft_util




# dir_data = '/mnt/photometry/AT2025oao/'
# f_sci=['field10743-2025-07-14']
# f_ref=['field10743-2025-12-14']

# dir_data = '/mnt/photometry/AT2025das/'
# f_sci=['field21719-2025-03-17']
# f_ref=['field21719-2025-12-13']

# dir_data = '/mnt/photometry/AT2024abxa/'
# f_sci=['field17406-2024-10-23']
# f_ref=['field17406-2026-01-11']
# detector lines

# dir_data = '/mnt/photometry/AT2025lqt/'
# f_sci=['field13402-2025-05-04']
# f_ref=['field13402-2025-05-06']

# dir_data = '/mnt/photometry/AT2024tti/'
# f_sci=['field18017-2024-09-02']
# f_ref=['field18017-2026-01-11']

# STATS for AT2025oos:
# dir_data='/mnt/photometry/AT2025oos/'
# f_sci = ['field14619-2025-05-15']
# f_ref = ['field14619-2026-01-12']

# # alignment is super weird
# dir_data='/mnt/photometry/AT2025trz/'
# f_sci = ['field13819-2025-06-13']
# f_ref = ['field13819-2025-12-14']

# dir_data='/mnt/photometry/AT2025vja/' # noisy diff image 
# f_sci = ['field13257-2025-09-01']
# f_ref = ['field13257-2025-12-13']

# bad reduction (crazy corners in sci), ref brighter than sci, try removing corners?
# corners removed: alignment is messed up
# dir_data = '/mnt/photometry/AT2024glc/'
# f_sci = ['field7711-2024-03-26']
# f_ref = ['field7711-2024-01-21']

# problem corner, noisy diff image
# dir_data = '/mnt/photometry/SN2025roy/'
# f_sci = ['field16046-2025-08-17']
# f_ref = ['field16046-2025-12-14']


# dir_data = '/mnt/photometry/AT2025zzj/'
# f_sci = ['field8218-2025-10-08']
# f_ref = ['field8218-2026-01-10']


# singular matrix, might be due to messing up the header params while trying to crop
# dir_data = '/mnt/photometry/AT2025whh/'
# f_sci = ['field13239-2025-09-01']
# f_ref = ['field13239-2025-12-14']

# blurry images? horrible subtraction
# dir_data = '/mnt/photometry/AT2025bob/'
# f_sci = ['field1256-2025-02-14']
# f_ref = ['field1256-2025-02-17']

# negative residuals
# dir_data = '/mnt/photometry/AT2025ably/' #ref brigher, long afterglow possible?
# f_sci = ['field12662-2025-10-16']
# f_ref = ['field12662-2026-01-11']



# dir_data = '/mnt/photometry/AT2025wsc/' #ref brigher, long afterglow possible?
# f_sci = ['field15121-2025-08-30']
# f_ref = ['field15121-2025-12-14']

# dir_data = '/mnt/photometry/AT2025btg/' # really dense
# f_ref = ['field1258-2025-02-16']
# f_sci = ['field1258-2025-02-13']


# dir_data = '/mnt/photometry/AT2025xfn/'
# f_ref = ['field9345-2025-12-13']
# f_sci = ['field9345-2025-09-03']

dir_data = '/mnt/photometry/SN2024xyu/'
f_sci = 'field17660-2024-10-22'
f_ref = 'field17660-2025-12-14'

# dir_data = '/mnt/photometry/SN2025adcv/'
# f_sci = ['field3036-2025-11-13']
# f_ref = ['field3036-2025-11-17']


# dir_data = '/mnt/photometry/AT2025lbx/' # weird alignment?
# f_sci = ['field16523-2025-06-01']
# f_ref = ['field16523-2026-01-12']

# dir_data = '/mnt/photometry/AT2024dpb/'
# f_sci = ['field10883-2024-03-14']
# f_ref = ['field10883-2026-01-12']

# dir_data = '/mnt/photometry/AT2024xwh/'
# f_sci = ['field10821-2024-10-09']
# f_ref = ['field10821-2024-10-05']

# dir_data = '/mnt/photometry/AT2025adki/' # crazy coadds
# f_sci = ['field5825-2025-11-17']
# f_ref = ['field5825-2025-11-13']


# dir_data = '/mnt/photometry/AT2025aayv/' # host galaxy, should have detected!
# f_sci = ['field8786-2025-11-03']
# f_ref = ['field8786-2026-01-11']

# dir_data = '/mnt/photometry/AT2024ztm/' 
# f_sci = ['field10519-2024-10-30']
# f_ref = ['field10519-2024-10-24']



# dir_data = '/mnt/photometry/AT2025lbx/' 
# f_sci = 'field16523-2025-06-01'
# f_ref = 'field16523-2026-01-12'


# dir_data = '/mnt/photometry/AT2025uvl/' # i see source but not sextracted
# f_sci = ['field10496-2025-09-02']
# f_ref = ['field10496-2025-12-13']

band = 'J'



/home/alex/miniconda3/envs/photo310/lib/python3.10/site-packages/sfft/utils/HoughDetection.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version


TabError: inconsistent use of tabs and spaces in indentation (photometrus_utils.py, line 198)

#### Parameters

In [ ]:

# sfft_util.SFFT(dir_data, f_sci[0], f_ref[0], band)

t = time.time()
FILE_BASENAME = f"sub_{t}"

# * computing backend and resourse 
BACKEND_4SUBTRACT = 'Numpy'	 # FIXME {'Cupy', 'Numpy'}, Use 'Numpy' if you only have CPUs
CUDA_DEVICE_4SUBTRACT = '0'	 # FIXME ONLY work for backend Cupy
NUM_CPU_THREADS_4SUBTRACT = 10   # FIXME ONLY work for backend Numpy

# * required info in FITS header
GAIN_KEY = 'GAIN'  # not GAIN_CAL			 # NOTE Keyword of Gain in FITS header
SATUR_KEY = 'SATURATE'		  # NOTE Keyword of Saturation in FITS header

# * how to subtract
GKerHW = None				   # FIXME given matching kernel half width
KerHWRatio = 2.0				# FIXME Ratio of kernel half width to FWHM (typically, 1.5-2.5).
KerPolyOrder = 2			  # FIXME {0, 1, 2, 3}, Polynomial degree of kernel spatial variation
BGPolyOrder = 0 # trivial for sparse (already sky subtracted)			# As above but for CROWDED field
ConstPhotRatio =  False	  #False	# FIXME Constant photometric ratio between images? dont scale them
PriorBanMask = None			 # FIXME None or a boolean array with same shape of science/reference.

COARSE_VAR_REJECTION = False #True	 # FIXME Coarse Variable Rejection? {True, False}
CVREJ_MAGD_THRESH = 0.12		# FIXME magnitude threshold for Coarse Variable Rejection
ELABO_VAR_REJECTION = False #True	  # FIXME Elaborate Variable Rejection? {True, False}

ref_stack = dir_data+f_ref+f'/{band}/stack'
sci_stack = dir_data+f_sci+f'/{band}/stack'



#### Alignment and preperation of files

In [ ]:
FITS_SCI, FITS_REF, ra, dec = sfft_util.get_sci_ref(sci_stack, ref_stack)
ForceConv =  sfft_util.get_force_conv(sci_stack,ref_stack)

sci_header=fits.getheader(FITS_SCI)
ref_header = fits.getheader(FITS_REF)

FITS_REF_OLD, FITS_SCI_OLD = FITS_REF, FITS_SCI
FITS_SCI, sci_bkg = sfft_util.remove_bkg_crop(FITS_SCI, "science_bkgsub.fits", sci_header, ra, dec)
FITS_REF, ref_bkg = sfft_util.remove_bkg_crop(FITS_REF, "reference_bkgsub.fits", ref_header, ra, dec)


# align with Orion's method
FITS_REF_OLD = FITS_REF
FITS_SCI, FITS_REF = util.multi_epoch_astrom(FITS_SCI, FITS_REF)

FITS_DIFF =dir_data+FILE_BASENAME+'_%s.sfftdiff.fits' %(pa.basename(FITS_SCI)[:-5])			# difference
FITS_REF_al = FITS_REF[:-5] + '.aligned.fits'   # refernce aligned


# align with swarp
PY_SWarp.PS(FITS_obj=FITS_REF, FITS_ref=FITS_SCI, FITS_resamp=FITS_REF_al, \
    GAIN_KEY=GAIN_KEY, SATUR_KEY=SATUR_KEY, OVERSAMPLING=1, RESAMPLING_TYPE='LANCZOS3', \
    SUBTRACT_BACK='N', FILL_VALUE=np.nan, VERBOSE_TYPE='NORMAL', VERBOSE_LEVEL=2)

print('\nMeLOn CheckPoint: IMAGE ALIGNMENT WITH SWARP DONE!\n')
print('Ref. Image aligned: '+FITS_REF_al)
print('Diff. Image:'+FITS_DIFF)

# get coordinates of sources to determine coords
all_coords = sfft_util.get_all_coords(sci_stack, ref_stack)



#### Subtraction

In [ ]:

ForceConv='AUTO' # shouldnt be
PixA_DIFF, SFFTPrepDict = Easy_SparsePacket.ESP(FITS_REF=FITS_REF_al, FITS_SCI=FITS_SCI,
                            FITS_DIFF=FITS_DIFF, FITS_Solution=None, ForceConv=ForceConv, GKerHW=GKerHW,
                            KerHWRatio=KerHWRatio, KerHWLimit=(5, 20), KerPolyOrder=KerPolyOrder, 
                            BGPolyOrder=BGPolyOrder, ConstPhotRatio=ConstPhotRatio, MaskSatContam=False, 
                            GAIN_KEY=GAIN_KEY, SATUR_KEY=SATUR_KEY, BACK_TYPE='MANUAL', BACK_VALUE=0.0, 
                            BACK_SIZE=64, BACK_FILTERSIZE=2, DETECT_THRESH=2, DETECT_MINAREA=5, 
                            DETECT_MAXAREA=0, DEBLEND_MINCONT=1e-4, BACKPHOTO_TYPE='LOCAL', 
                            ONLY_FLAGS=[0], BoundarySIZE=30, XY_PriorSelect=all_coords, PointSource_MINELLIP=0.3, MatchTol=None, 
                            MatchTolFactor=3.0, StarExt_iter=4, XY_PriorBan=None,
                            PostAnomalyCheck=False, PAC_RATIO_THRESH=5.0, BACKEND_4SUBTRACT=BACKEND_4SUBTRACT, 
                            CUDA_DEVICE_4SUBTRACT=CUDA_DEVICE_4SUBTRACT,
                            NUM_CPU_THREADS_4SUBTRACT=NUM_CPU_THREADS_4SUBTRACT)[:2]

# (2, 20)

# MAG_OFFSET = -2.5 * np.log10(ConstPhotRatio) # 0.5
# SFFTPrepDict["MAG_OFFSET"] = MAG_OFFSET



  
# PixA_DIFF, SFFTPrepDict = Easy_SparsePacket.ESP(FITS_REF=FITS_REF_al, FITS_SCI=FITS_SCI, \
# 								 FITS_DIFF=FITS_DIFF, FITS_Solution=None, ForceConv=ForceConv, GKerHW=None, \
# 								 KerHWRatio=KerHWRatio, KerHWLimit=(2, 20), KerPolyOrder=KerPolyOrder, \
# 								 BGPolyOrder=BGPolyOrder, ConstPhotRatio=ConstPhotRatio, MaskSatContam=False, \
# 								 GAIN_KEY=GAIN_KEY, SATUR_KEY=SATUR_KEY, BACK_TYPE='MANUAL', BACK_VALUE=0.0, \
# 								 BACK_SIZE=64, BACK_FILTERSIZE=2, DETECT_THRESH=2, DETECT_MINAREA=5, \
# 								 DETECT_MAXAREA=0, DEBLEND_MINCONT=1e-4, BACKPHOTO_TYPE='LOCAL', \
# 								 ONLY_FLAGS=[0], BoundarySIZE=30, XY_PriorSelect=None, Hough_MINFR=0.1, \
# 								 Hough_PeakClip=0.7, BeltHW=0.2, PointSource_MINELLIP=0.3, MatchTol=None, \
# 								 MatchTolFactor=3.0, COARSE_VAR_REJECTION=COARSE_VAR_REJECTION, \
# 								 CVREJ_MAGD_THRESH=CVREJ_MAGD_THRESH, ELABO_VAR_REJECTION=ELABO_VAR_REJECTION, \
# 								 EVREJ_RATIO_THREH=5.0, EVREJ_SAFE_MAGDEV=0.04, StarExt_iter=4, XY_PriorBan=None, \
# 								 PostAnomalyCheck=False, PAC_RATIO_THRESH=5.0, BACKEND_4SUBTRACT=BACKEND_4SUBTRACT, \
# 								 CUDA_DEVICE_4SUBTRACT=CUDA_DEVICE_4SUBTRACT, \
# 								 NUM_CPU_THREADS_4SUBTRACT=NUM_CPU_THREADS_4SUBTRACT)[:2]

# util.display(FITS_SCI)
# util.display(FITS_REF)
# util.display(FITS_DIFF)

# util.make_cutout(FITS_SCI, ra, dec, png=True)
# util.make_cutout(FITS_REF, ra, dec, png=True)
# util.make_cutout(FITS_DIFF, ra, dec, png=True)


# sfft_util.get_diff_stats(SFFTPrepDict, PixA_DIFF)


In [ ]:
util.display(FITS_SCI)
util.display(FITS_REF)
util.display(FITS_DIFF)

# util.make_cutout(FITS_SCI, ra, dec, png=True)
# util.make_cutout(FITS_REF, ra, dec, png=True)
# util.make_cutout(FITS_DIFF, ra, dec, png=True)


sfft_util.get_diff_stats(SFFTPrepDict, PixA_DIFF)

In [ ]:

sources = sfft_util.source_extract(FITS_DIFF)


#TODO add a check here to see if source ra, dec in sources 
good_ztf_sources = sfft_util.ztf_cuts(sources, PixA_DIFF)

print(np.shape(PixA_DIFF))
print(len(good_ztf_sources), "sources")
print(ra, dec)




In [ ]:
diff_savename, diff_threshname = util.make_cutout(FITS_SCI, ra, dec, png=True, display_file=True, name_ext="sci")
diff_savename, diff_threshname = util.make_cutout(FITS_REF, ra, dec, png=True, display_file=True, name_ext="ref")
diff_savename, diff_threshname = util.make_cutout(FITS_DIFF, ra, dec, png=True, display_file=True, name_ext="diff")




In [ ]:
source_savename, source_threshname, closest_ra, closest_dec = sfft_util.closest_source(good_ztf_sources, FITS_DIFF, ra, dec)

distance = ((ra-closest_ra)**2 + (dec-closest_dec)**2)**(1/2)
distance = round(distance*3600,3)

threshold = 60 / 3600
for i in range(len(good_ztf_sources)):
   
    source = good_ztf_sources.iloc[i]
    source_ra = source['ALPHA_J2000']
    source_dec = source['DELTA_J2000']
    if abs(source_ra-ra) < threshold and abs(source_dec-dec) < threshold:
        print("Found source!")
        print(source_ra, source_dec)
        print(source)
		  
    
print("distance to closest extracted source:", distance)

util.display(f"{source_savename}.png", source_savename)

In [ ]:
sci_cat = sfft_util.source_extract(FITS_SCI)
# print(sci_cat)
ref_cat = sfft_util.source_extract(FITS_REF)
# print(ref_cat)

In [ ]:
print("Sources in sci:",len(sci_cat))
print("Sources in ref:",len(ref_cat))
print("Sources in dif:",len(sources))


print("Sources in sci:",len(sci_cat))
print("Sources in ref:",len(ref_cat))
print("Sources in dif:",len(sources))